# 03 · Agent null control — does it confabulate, and does it track the evidence?

The question almost nobody tests. Three arms over the matched MPRA benchmark:
1. **Negatives** (should decline) — does the agent invent a mechanism on non-functional
   variants, or correctly report low confidence?
2. **Random positives** — engine-quiet functional variants (agent should still not
   over-assert).
3. **Strong-signal positives** (`|ChromBPNet Δ|` top) — engine *fires*: does it assert?

Together these close the biconditional: **the agent names a mechanism iff the engine fires,
and confabulates never.** Needs the sequence signals (hence the scorer) + an API key.

In [ ]:
# --- Setup: clone RegLens from GitHub + install -----------------------------
# No zip upload — clones the public repo fresh, so it's always current. Re-running
# this cell fast-forwards to the latest commit on main.
import os
if not os.path.isdir("/content/RegLens/.git"):
    !git clone --depth 1 https://github.com/kpal002/RegLens.git /content/RegLens
%cd /content/RegLens
!git pull -q --ff-only 2>/dev/null || echo "(using the freshly cloned snapshot)"
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > GPU.'
!pip -q install 'tensorflow>=2.12' pyfaidx typer numpy matplotlib
!pip -q install -e .
!echo "RegLens @ $(git rev-parse --short HEAD) | cwd $(pwd)"

In [ ]:
# --- Reference genome (hg38) + pretrained K562 ChromBPNet model -------------
import glob, os
os.makedirs('/content/ref', exist_ok=True)
%cd /content/ref
# hg38 via the Broad public bucket: resolves on Colab, chr-named, ships its .fai.
!wget -c -O hg38.fa     https://storage.googleapis.com/gcp-public-data--broad-references/hg38/v0/Homo_sapiens_assembly38.fasta
!wget -c -O hg38.fa.fai https://storage.googleapis.com/gcp-public-data--broad-references/hg38/v0/Homo_sapiens_assembly38.fasta.fai
# ENCODE K562 ATAC ChromBPNet models (5 folds, bias-corrected).
![ -f ENCFF984RAF.tar.gz ] || wget -q -c -O ENCFF984RAF.tar.gz https://www.encodeproject.org/files/ENCFF984RAF/@@download/ENCFF984RAF.tar.gz
!mkdir -p encode_models && tar -xzf ENCFF984RAF.tar.gz -C encode_models
n = len(glob.glob('encode_models/**/*chrombpnet_nobias*.h5', recursive=True))
print(n, 'K562 folds | genome:', os.path.exists('hg38.fa'))
%cd /content/RegLens

In [ ]:
# --- Reasoning layer: Anthropic SDK + API key (from a Colab secret) ---------
# Add your key once as a Colab secret named ANTHROPIC_API_KEY (🔑 panel in the left
# sidebar, then toggle "Notebook access" on) — it never appears in the notebook.
# Falls back to a one-time hidden prompt if the secret isn't set.
!pip -q install anthropic
import os
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")
assert os.environ["ANTHROPIC_API_KEY"].startswith("sk-"), "set your key above"

In [ ]:
# Fold + reverse-complement averaged K562 backend (the validated engine).
from reglens.tools.chrombpnet_score import ChromBPNetScorer, load_backend
scorer = ChromBPNetScorer(load_backend('/content/ref/encode_models', average_rc=True),
                          window_length=2114, model_name='K562-5fold+rc')
GENOME = '/content/ref/hg38.fa'

In [ ]:
# --- Arms 1 & 2: paired control (negatives should decline, positives asserted) ---
from reglens.agents.multi_agent import MultiAgentInterpreter
from reglens.validation.null_control import run_paired_control, render_paired
HEMA = ["BCL11A","HBB","HBG1","PKLR-48h","PKLR-24h","GP1BA"]
neg, pos = run_paired_control('/content/RegLens/data/benchmarks/kircher_mpra_grch38.tsv', MultiAgentInterpreter(),
    n_neg=8, n_pos=8, seed=7, elements=HEMA, celltype='K562',
    genome_path=GENOME, scorer=scorer, progress=True)
print(render_paired(neg, pos))

In [ ]:
# --- Arm 3: strong-signal positive control (force the engine to fire) --------
from reglens.validation.null_control import run_strong_positive_control, render_summary
strong = run_strong_positive_control('/content/RegLens/data/benchmarks/kircher_mpra_grch38.tsv', MultiAgentInterpreter(),
    scorer=scorer, genome_path=GENOME, n=8, pool=200, seed=7, elements=HEMA,
    celltype='K562', progress=True)
print(render_summary(strong, 'Strong-signal positive control'))
for o in strong:
    print('\n', o.variant, o.element, '->', o.verdict, '| conf', o.interpretation.confidence,
          '| tf', o.interpretation.tf); print('  ', o.interpretation.mechanism)

## Honest reading

Zero confabulations across the three arms (rule of three → 95% upper bound ≈12%). The
agent names a mechanism **iff** the motif channel fires with a concordant Δ; confidence is
corroboration-gated and capped for these synthetic MPRA variants. Full write-up: `RESULTS.md`
§ "Agent null control".